# Task 3: Multi-Tool Autonomous AI Agent

**Project:** CartSense, a purchase-decision agent
**Language and stack:** Python 3.12, LangChain tool-calling agent (`create_agent`), Pydantic, OpenRouter (`openai/gpt-oss-20b`), Tavily and DuckDuckGo web search, Gradio (optional UI), pytest, ruff, uv

This notebook walks through my Task 3 project from top to bottom. It does not copy the code. It imports the `cartsense` package, shows the source of each module, and runs the real agent, so what you read here is the same code the tests check.

## Contents

1. Objective and key concepts
2. Setup
3. Architecture and tools
4. Source code tour (config, models, tools, scoring, agent, cli)
5. The system prompt
6. The Pydantic wishlist model
7. The scoring functions
8. Live run: the agent researches a purchase
9. The saved recommendation
10. Safeguards
11. Interactive mode and the web UI
12. Tests
13. Design decisions, limitations, skills

## 1. Objective and key concepts

**Objective:** build an autonomous AI agent that chooses and uses several tools to complete a task. CartSense helps you decide what to buy. You type a request such as `I want to buy a dishwasher`. The agent reads or builds your wishlist, searches the web for current prices and specs, calculates running costs and a value score in Python, prints its reasoning chain, and saves a recommendation to `output/<category>.md`.

**Key concepts**

- LangChain tool-calling agent (`langchain.agents.create_agent`) with five tools
- Multi-tool task execution, with the reasoning chain (Thought, Action, Observation) printed live
- Web search tool: Tavily when a key is set, otherwise DuckDuckGo, with retry and backoff
- Calculator tool: safe arithmetic parsed with `ast`, never `eval`, plus running cost, value score and total cost of ownership
- Local file tools: reading and saving a wishlist validated by a Pydantic model
- Guardrails enforced in code, not left to the model: step and search caps, duplicate-call detection, a final-answer gate

## 2. Setup

In [1]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

SKIP_FOLDERS = {".venv", "venv", "node_modules", ".git", "__pycache__", ".ipynb_checkpoints"}


def find_project_root() -> Path:
    """Find the folder that contains the cartsense package.

    Works when the notebook sits in the repository root, inside the CartSense
    project folder, or in any folder in between.
    """
    here = Path.cwd().resolve()

    # 1. This folder or one of its parents (notebook opened inside the project).
    for folder in [here, *here.parents]:
        if (folder / "cartsense" / "cli.py").exists():
            return folder

    # 2. A project folder below this one (notebook opened from the repository root).
    for current, subfolders, _ in os.walk(here):
        depth = len(Path(current).relative_to(here).parts)
        subfolders[:] = (
            [d for d in subfolders if d not in SKIP_FOLDERS and not d.startswith(".")]
            if depth < 3 else []
        )
        if (Path(current) / "cartsense" / "cli.py").exists():
            return Path(current)

    raise FileNotFoundError(
        "Could not find the cartsense package (a folder named 'cartsense' containing cli.py). "
        "Start Jupyter from the repository root or from the CartSense project folder."
    )


STARTED_IN = Path.cwd().resolve()
PROJECT = find_project_root()
os.chdir(PROJECT)  # wishlists, output and .env paths below are relative to the project folder
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print("Notebook started in:", STARTED_IN.name)
print("Project folder found:", PROJECT.name)
print("Python:", sys.version.split()[0])

Notebook started in: Datheon_Generative_AI_Internship
Project folder found: Task3_CartSense
Python: 3.12.12


In [2]:
# Check which keys are configured, without printing any secret.
env_file = PROJECT / ".env"
print(".env file found:", env_file.exists())
if env_file.exists():
    names = set()
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            names.add(line.split("=", 1)[0].strip())
    print("OPENROUTER_API_KEY present:", "OPENROUTER_API_KEY" in names)
    print("Search provider:", "Tavily (TAVILY_API_KEY is set)" if "TAVILY_API_KEY" in names else "DuckDuckGo (no Tavily key)")

.env file found: True
OPENROUTER_API_KEY present: True
Search provider: Tavily (TAVILY_API_KEY is set)


This notebook can live in the repository root or inside the CartSense project folder. The first cell finds the project folder by itself and switches into it, so the wishlist, output and `.env` paths all resolve correctly. Start Jupyter from the repository root with `uv run --with jupyter jupyter lab`.

## 3. Architecture and tools

```
request --> agent loop (LangChain create_agent) --> final-answer gate --> output/<category>.md
              |  Thought / Action / Observation printed live
              |-- read_wishlist   saved wishlist for the category (local file reader)
              |-- ask_user        interview, max 10 questions (interactive only)
              |-- save_wishlist   validated JSON, wishlists/ only (interactive only)
              |-- web_search      Tavily (if a key is set) or DuckDuckGo, with retry and backoff
              |-- calculator      safe arithmetic, running_cost, value_score, tco
```

| File | Job |
|---|---|
| `cartsense/config.py` | Environment settings, validated at startup |
| `cartsense/models.py` | Pydantic `Wishlist` model with constraints |
| `cartsense/tools.py` | The tools, input guards, retry, `ToolError` |
| `cartsense/scoring.py` | Pure functions: `running_cost()`, `value_score()`, `tco()` |
| `cartsense/agent.py` | Tool wiring, agent loop, step trace, final-answer gate |
| `cartsense/cli.py` | Command line, saving the recommendation |
| `prompts/system.md` | The system prompt |

## 4. Source code tour

The helper below prints the source of a module straight from the package, so this notebook always matches the real code.

In [3]:
import importlib
import inspect


def show_source(module_name: str) -> None:
    try:
        module = importlib.import_module(module_name)
        source = inspect.getsource(module)
    except Exception as error:  # keep "Run All Cells" going if one module cannot be shown
        print(f"Could not show {module_name}: {error}")
        return
    print(f"# {module_name}  ({len(source.splitlines())} lines)\n")
    print(source)

### 4.1 Configuration (`config.py`)

Settings from `.env` or the environment: model name, base URL, search region (default `in-en`) and search provider. Validated at startup, so a mistake stops the program with a clear message.

In [4]:
show_source("cartsense.config")

# cartsense.config  (101 lines)

"""Environment-driven configuration for CartSense."""

import os
import re
from dataclasses import dataclass, field
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DEFAULT_BASE_URL = "https://openrouter.ai/api/v1"
DEFAULT_MODEL = "openai/gpt-oss-20b"
DEFAULT_MAX_STEPS = 40
DEFAULT_TEMPERATURE = 0.0
DEFAULT_SEARCH_REGION = "in-en"
REGION_PATTERN = r"^[a-z]{2}-[a-z]{2}$"


class ConfigError(Exception):
    """Raised when required configuration is missing or invalid."""


@dataclass(frozen=True)
class Config:
    """Validated settings, built once at startup."""

    api_key: str
    base_url: str
    model: str
    max_steps: int
    temperature: float
    search_region: str = DEFAULT_SEARCH_REGION
    search_provider: str = "ddgs"
    tavily_api_key: str = field(default="", repr=False)


def _read_int(name: str, default: int) -> int:
    """Read an integer env var; raise ConfigError if it is 

### 4.2 Wishlist model (`models.py`)

The Pydantic `Wishlist` model with constraints. Anything the agent saves to disk goes through it first.

In [5]:
show_source("cartsense.models")

# cartsense.models  (58 lines)

"""Pydantic models for the saved wishlist."""

import re
from typing import Self

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

MAX_LIST_ITEMS = 10
MAX_ITEM_LENGTH = 80
CATEGORY_PATTERN = r"^[a-z0-9][a-z0-9 _-]{0,39}$"


def clean_category(raw: str) -> str:
    """Normalize a category name: lowercase, trimmed, single spaces."""
    return re.sub(r"\s+", " ", raw.strip().lower())


class Wishlist(BaseModel):
    """What the buyer wants, validated before it is saved or scored."""

    model_config = ConfigDict(extra="forbid")

    category: str = Field(pattern=CATEGORY_PATTERN)
    budget: float = Field(gt=0, le=10_000_000, description="Maximum spend in local currency.")
    currency: str = Field(default="INR", pattern=r"^[A-Z]{3}$", description="Currency code.")
    years_of_use: float = Field(gt=0, le=50, description="Expected years of ownership.")
    warranty_years: float = Field(ge=0, le=20, description="Minimum 

### 4.3 Tools (`tools.py`)

The five tools, their input guards, the search retry with backoff, and `ToolError`. The calculator is parsed with `ast`, with caps on length, exponent and result.

In [6]:
show_source("cartsense.tools")

# cartsense.tools  (320 lines)

"""Tools the agent can call: calculator, wishlist reader and web search."""

import ast
import logging
import operator
import random
import re
import time
from collections.abc import Callable
from pathlib import Path
from typing import Any

from pydantic import ValidationError

from cartsense.config import DEFAULT_SEARCH_REGION, PROJECT_ROOT
from cartsense.models import CATEGORY_PATTERN, Wishlist, clean_category

logger = logging.getLogger(__name__)

WISHLIST_DIR = PROJECT_ROOT / "wishlists"
MAX_WISHLIST_BYTES = 10_000
MAX_EXPRESSION_LENGTH = 200
MAX_EXPONENT = 100
MAX_ABS_RESULT = 1e15
MAX_QUERY_LENGTH = 200
MAX_SNIPPET_LENGTH = 600
MAX_RESULTS_LIMIT = 10
MIN_RESULTS_LIMIT = 3
SEARCH_MAX_ATTEMPTS = 3
SEARCH_BASE_DELAY = 1.0

SearchFn = Callable[[str, int], list[dict[str, str]]]


class ToolError(Exception):
    """A tool failed in a way the agent can read and recover from."""


# --- calculator -----------------------------------------------------------

### 4.4 Scoring (`scoring.py`)

Pure functions for running cost, value score and total cost of ownership. The numbers come from Python, not from the model.

In [7]:
show_source("cartsense.scoring")

# cartsense.scoring  (241 lines)

"""Pure scoring functions: total cost of ownership and value score."""

from statistics import mean
from typing import Any

WEIGHT_PRIORITIES = 0.50
WEIGHT_BUDGET = 0.30
WEIGHT_WARRANTY = 0.20
MAX_RATING = 10.0
WEEKS_PER_YEAR = 52
MONTHS_PER_YEAR = 12
DAYS_PER_YEAR = 365


def _require_non_negative(**values: float) -> None:
    """Raise ValueError naming the first argument that is negative."""
    for name, value in values.items():
        if value < 0:
            raise ValueError(f"{name} must not be negative, got {value}")


def tco(
    price: float,
    years_of_use: float,
    yearly_running_cost: float = 0.0,
    yearly_repair_cost: float = 0.0,
    warranty_years: float = 0.0,
    resale_value: float = 0.0,
    yearly_consumables_cost: float = 0.0,
) -> float:
    """Total cost of ownership over the years of use.

    yearly_running_cost is energy and water; yearly_consumables_cost is things like
    detergent, cleaning liquid, filters. Repairs

### 4.5 Agent (`agent.py`)

Wires the tools into a LangChain agent, runs the loop, records the step trace and applies the final-answer gate.

In [8]:
show_source("cartsense.agent")

# cartsense.agent  (874 lines)

"""The LangChain tool-calling agent and its step trace."""

import functools
import json
import logging
import re
import time
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import openai
from langchain.agents import create_agent
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import BaseTool, StructuredTool
from langchain_openai import ChatOpenAI
from langgraph.errors import GraphRecursionError
from pydantic import BaseModel, Field

from cartsense.config import PROJECT_ROOT, Config
from cartsense.scoring import ownership_cost, running_cost, tco, value_score
from cartsense.tools import (
    WISHLIST_DIR,
    ToolError,
    UserPrompter,
    calculator,
    is_product_url,
    read_wishlist,
    save_wishlist,
    web_search,
)

logger = logging.getLo

### 4.6 Command line (`cli.py`)

Parses the request, runs the agent interactively or with `--no-interactive`, and saves the recommendation. Exit codes: 0 done, 1 failed or step limit reached, 2 configuration error, 130 cancelled.

In [9]:
show_source("cartsense.cli")

# cartsense.cli  (320 lines)

"""Command-line entry point: run the agent and save the recommendation."""

import argparse
import json
import logging
import re
import sys
from collections.abc import Callable, Sequence
from pathlib import Path
from typing import Any, TextIO

from langchain_core.language_models.chat_models import BaseChatModel

from cartsense.agent import (
    AgentError,
    AgentResult,
    Requirements,
    Step,
    build_model,
    build_requirements,
    build_tools,
    format_step,
    load_system_prompt,
    mode_note,
    run_agent,
)
from cartsense.config import PROJECT_ROOT, Config, ConfigError, load_config
from cartsense.models import CATEGORY_PATTERN, clean_category
from cartsense.tools import WISHLIST_DIR, UserPrompter, is_product_url

OUTPUT_DIR = PROJECT_ROOT / "output"
LOG_DIR = PROJECT_ROOT / "logs"
LOG_FORMAT = "%(asctime)s %(levelname)s %(name)s %(message)s"
MAX_DISPLAY_CHARS = 400
FALLBACK_CATEGORY = "recommendation"
EXIT_OK = 0
EXIT_FAILED = 1
EXIT

## 5. The system prompt

The instructions for the agent live in a file, not inside a Python string.

In [10]:
prompt_path = PROJECT / "prompts" / "system.md"
if prompt_path.exists():
    print(prompt_path.read_text(encoding="utf-8"))
else:
    print("Prompt file not found at prompts/system.md")

You are CartSense, a purchase-decision agent. You help one buyer choose what to buy.
Use only the tools you are given. Never invent a tool.

## Workflow
1. Call read_wishlist with the product category from the request.
2. If ask_user is available:
   - If a saved wishlist exists, show it and ask whether to reuse or update it.
   - If none exists, interview the buyer with 6 to 10 questions, never more than 10:
     budget and its currency, years of use, warranty wanted, top two priorities, dealbreakers, plus 1 or 2
     category-specific specs (for a washing machine: loads per week and household size).
   - Then call save_wishlist with the answers, if it is available.
3. If ask_user is not available (non-interactive run), use the saved wishlist. If there is
   none, use sensible defaults and state each one as an assumption.
4. Pick exactly 3 candidate models. Make at most 2 web_search calls per candidate: one
   for the price in the buyer's currency and country, one for reviews and spec

## 6. The Pydantic wishlist model

A wishlist describes what the user wants. It is validated before it is written to `wishlists/`. The cell below lists the model's fields, loads the bundled sample, and then shows which fields reject a clearly invalid value.

In [11]:
from pydantic import BaseModel, ValidationError
from cartsense.models import Wishlist

fields = getattr(Wishlist, "model_fields", None) or getattr(Wishlist, "__fields__", {})
print("Wishlist fields:", ", ".join(fields))

sample_path = PROJECT / "data" / "synthetic" / "wishlists" / "phone.json"
sample = json.loads(sample_path.read_text(encoding="utf-8")) if sample_path.exists() else None


def validate(data):
    if hasattr(Wishlist, "model_validate"):
        return Wishlist.model_validate(data)
    return Wishlist.parse_obj(data)


if sample is None:
    print("Sample wishlist not found at data/synthetic/wishlists/phone.json")
else:
    print("\nSample wishlist:")
    print(json.dumps(sample, indent=2, ensure_ascii=False)[:1500])
    validate(sample)
    print("\nThe sample passes validation.")

Wishlist fields: category, budget, currency, years_of_use, warranty_years, priorities, dealbreakers, specs

Sample wishlist:
{
  "category": "phone",
  "budget": 25000,
  "currency": "INR",
  "years_of_use": 3,
  "warranty_years": 1,
  "priorities": [
    "battery life",
    "camera quality"
  ],
  "dealbreakers": [
    "no fast charging"
  ],
  "specs": {
    "ram": "8 GB",
    "storage": "128 GB"
  }
}

The sample passes validation.


In [12]:
# Replace each top-level field with -1 and see which fields refuse it.
if sample:
    rejected = []
    for key in sample:
        broken = {**sample, key: -1}
        try:
            validate(broken)
        except ValidationError:
            rejected.append(key)
    print("Fields that reject the value -1:", ", ".join(rejected) if rejected else "none")
    try:
        validate({})
    except ValidationError as error:
        print("\nAn empty wishlist is rejected:", error.errors()[0].get("msg"))
    else:
        print("\nAn empty wishlist is accepted (every field has a default).")

Fields that reject the value -1: category, budget, currency, years_of_use, warranty_years, priorities, dealbreakers, specs

An empty wishlist is rejected: Field required


## 7. The scoring functions

The price comparison is done by plain Python functions, so the model never does arithmetic in its head. The cell shows the signature and the docstring of each one.

In [13]:
import cartsense.scoring as scoring

for name in ("running_cost", "value_score", "tco"):
    function = getattr(scoring, name, None)
    if function is None:
        print(f"{name}: not found")
        continue
    print(f"{name}{inspect.signature(function)}")
    print("   ", (inspect.getdoc(function) or "(no docstring)").splitlines()[0])

running_cost(uses_per_week: float, energy_cost_per_use: float = 0.0, water_cost_per_use: float = 0.0, consumable_cost_per_use: float = 0.0, other_yearly_cost: float = 0.0, energy_kwh_per_use: float = 0.0, electricity_rate_per_kwh: float = 0.0, water_litres_per_use: float = 0.0, water_rate_per_litre: float = 0.0) -> dict[str, float]
    Running cost per use, day, week, month and year (purchase price excluded).
value_score(price: float, budget: float, warranty_offered: float, warranty_wanted: float, priority_ratings: list[float], dealbreaker_hit: bool = False) -> float
    Score an option from 0 to 100 against the buyer's wishlist.
tco(price: float, years_of_use: float, yearly_running_cost: float = 0.0, yearly_repair_cost: float = 0.0, warranty_years: float = 0.0, resale_value: float = 0.0, yearly_consumables_cost: float = 0.0) -> float
    Total cost of ownership over the years of use.


## 8. Live run: the agent researches a purchase

This runs the real agent in non-interactive mode. It copies the sample phone wishlist into `wishlists/`, searches the web, calls the calculator, prints its reasoning chain and writes `output/phone.md`. It needs a valid `OPENROUTER_API_KEY` and internet access. One run can use up to 12 searches (8 general, 4 store searches), and the Tavily free tier is credit-limited.

In non-interactive mode `ask_user` and `save_wishlist` are removed. The agent uses the saved wishlist or states its defaults.

In [14]:
wishlist_dir = PROJECT / "wishlists"
wishlist_dir.mkdir(exist_ok=True)
if sample_path.exists() and not (wishlist_dir / "phone.json").exists():
    shutil.copy(sample_path, wishlist_dir / "phone.json")
print("Wishlist ready:", (wishlist_dir / "phone.json").exists())

Wishlist ready: True


In [15]:
def run_cartsense(*args: str, timeout: int = 900) -> subprocess.CompletedProcess:
    return subprocess.run(
        [sys.executable, "-m", "cartsense.cli", *args],
        capture_output=True, text=True, cwd=PROJECT, timeout=timeout,
    )


result = run_cartsense("--no-interactive", "I want to buy a phone")
print("exit code:", result.returncode)
print(result.stdout[-6000:])
print(result.stderr[-1500:])

exit code: 0
 leave it out and write 'price not found' for it in the answer.
Thought: No specific accessory prices were found in the search results. Let me recalculate without accessories.
Action: calculator({"arguments": {"accessories": [], "consumables": [], "electricity_rate_per_kwh": 7, "energy_kwh_per_use": 0.01, "other_yearly_cost": 0, "product_price": 24230, "uses_per_week": 7, "water_litres_per_use": 0, "water_rate_per_litre": 0.03}, "function": "ownership_cost", "label": "Samsung Galaxy F36 5G"})
Action: calculator({"arguments": {"accessories": [], "consumables": [], "electricity_rate_per_kwh": 7, "energy_kwh_per_use": 0.01, "other_yearly_cost": 0, "product_price": 23749, "uses_per_week": 7, "water_litres_per_use": 0, "water_rate_per_litre": 0.03}, "function": "ownership_cost", "label": "OPPO K13 5G"})
Action: calculator({"arguments": {"accessories": [], "consumables": [], "electricity_rate_per_kwh": 7, "energy_kwh_per_use": 0.01, "other_yearly_cost": 0, "product_price": 22999

Exit codes: `0` done, `1` failed or step limit reached, `2` configuration error, `130` cancelled. A run can fail the final-answer gate and exit with `1` instead of producing an answer. That is intended: the gate rejects an answer it cannot support, and running the cell again usually works.

## 9. The saved recommendation

The recommendation is saved to `output/<category>.md`. The saved file ends with every successful calculator call, verbatim, so each number can be traced back to a tool result.

In [16]:
outputs = sorted((PROJECT / "output").glob("*.md"), key=lambda p: p.stat().st_mtime) if (PROJECT / "output").exists() else []
if outputs:
    latest = outputs[-1]
    text = latest.read_text(encoding="utf-8")
    print("Saved file:", latest.name, f"({len(text.splitlines())} lines)\n")
    print(text[:6000])
else:
    print("No recommendation in output/ yet. Check the run above for errors.")

Saved file: phone.md (121 lines)

# CartSense recommendation: phone

Request: I want to buy a phone

# Phone Purchase Recommendation

Based on your wishlist (budget: ₹25,000, priorities: design/sleek look, dealbreaker: fast charging, specs: 8GB RAM/128GB storage), here are my top 3 recommendations:

## Ranked List

### 1. Samsung Galaxy F36 5G (8GB+128GB) - **₹24,230** ⭐ RECOMMENDED
- **Design**: Vegan leather finish, ultra-slim 7.7mm thickness — best matches your "sleek look" priority
- **Fast Charging**: 25W wired (meets dealbreaker ✓)
- **Display**: 6.7" FHD+ Super AMOLED, 120Hz
- **Camera**: 50MP OIS + 8MP ultrawide + 2MP macro, 13MP front
- **Battery**: 5000mAh
- **Warranty**: 1 year manufacturer
- **Buy Link**: [Amazon](https://www.amazon.in/Samsung-Galaxy-F36-Black-128/dp/B0FT834QNK)

### 2. OPPO K13 5G (8GB+128GB) - **₹23,749**
- **Design**: Sleek mix of gloss and matte finishes
- **Fast Charging**: 80W SuperVOOC (fastest in this list)
- **Display**: 6.67" AMOLED, 120Hz
- **Cam

## 10. Safeguards

Small models skip steps and invent numbers, so the rules that matter are enforced in code:

- **Safe calculator:** parsed with `ast`, never `eval`, with caps on length, exponent and result.
- **`save_wishlist`** writes only `wishlists/<cleaned category>.json`, validated and size-capped. No tool can move, edit or delete other files.
- **Caps:** at most 10 questions and 12 searches (8 general, 4 store searches).
- **Duplicate calls** return a short note instead of running again.
- **Final-answer gate:** an answer is rejected, and the model is sent back to fix it, if it skips required steps, cites a URL that no search returned, lacks a product-page link, or quotes a price or table score that no tool result supports.
- **Retries:** searches retry with backoff, and the model client retries transient API errors.
- **Calculator log:** the saved file ends with every successful calculator call, verbatim.

The code for these rules is in `tools.py` and `agent.py`, shown in sections 4.3 and 4.5.

## 11. Interactive mode and the web UI

Interactive mode asks questions, so it is run in a terminal rather than in this notebook. From the project folder:

```bash
uv run python -m cartsense.cli "I want to buy a phone"
```

If a wishlist exists, the agent asks "Reuse or update?". Otherwise it interviews you with 6 to 10 questions, saves your answers, and then researches.

**Optional web UI.** A Gradio front end for the same agent is in `ui/app.py`. It is presentation only, so the agent code is unchanged:

```bash
uv run --with "gradio>=5,<6" python ui/app.py
```

Open the local URL that Gradio prints (usually http://127.0.0.1:7860). It reads the same `.env` settings as the command line.

## 12. Tests

In [17]:
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    capture_output=True, text=True, cwd=PROJECT,
)
print(tests.stdout[-1500:])
print(tests.stderr[-500:])

....................................................................... [ 67%]
......................................................................   [100%]
214 passed in 1.35s




All tests use scripted fake models and fake search functions. No test touches the network or spends search credits.

## 13. Design decisions, limitations, skills

**Design decisions**

- **Rules in code, not in the prompt.** Models skip steps and invent numbers, so required steps, caps, the calculator limits and the final-answer gate are checked by Python.
- **Calculations in Python.** Running cost, value score and total cost of ownership come from pure functions in `scoring.py`, which are easy to test.
- **Minimal file access.** The only file the agent can write is a validated wishlist in `wishlists/`.
- **A tool-calling agent, scripted in tests.** Fake models and fake search functions make the tests repeatable and free.

**Limitations**

- Checks confirm that a number appears in the search results, not that it belongs to the model being discussed, so a price or spec can be attributed to the wrong product.
- Specs, ratings, warranty and dealbreaker claims are not verified against sources.
- A model with no product page in the results can still end up with a listing-page link.
- Prices and stock change, and links may go out of stock.
- Results depend on the search provider and on the model's discipline. A run can fail the gate and exit with code 1 instead of producing an answer.
- "Thought" lines appear only when the model writes text before a tool call.

**Skills shown in this project:** Python, LangChain, tool-calling agents, autonomous agent design, prompt engineering, web search integration (Tavily, DuckDuckGo), safe expression evaluation with `ast`, Pydantic validation, guardrails and input validation, retry and exponential backoff, error handling, Gradio, CLI design, pytest with scripted fakes, ruff, uv.